In [1]:
!pip install google-cloud-bigquery db-dtypes -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving phonic-chemist-509603-v9-6d0523a7b77d.json to phonic-chemist-509603-v9-6d0523a7b77d.json


In [3]:
from google.cloud import bigquery

key_filename = list(uploaded.keys())[0]  # grabs the uploaded file's name automatically
client = bigquery.Client.from_service_account_json(key_filename)
print("Connected to project:", client.project)

Connected to project: phonic-chemist-509603-v9


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

sales = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sales_train_validation.csv")
calendar = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/calendar.csv")
prices = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/COLAB DATA/sell_prices.csv")

dataset_id = "m5_raw"

for name, df in [("sales_train_validation", sales), ("calendar", calendar), ("sell_prices", prices)]:
    table_id = f"{client.project}.{dataset_id}.{name}"
    job = client.load_table_from_dataframe(df, table_id)
    job.result()
    print(f"Loaded {name} -> {table_id}")

Loaded sales_train_validation -> phonic-chemist-509603-v9.m5_raw.sales_train_validation
Loaded calendar -> phonic-chemist-509603-v9.m5_raw.calendar
Loaded sell_prices -> phonic-chemist-509603-v9.m5_raw.sell_prices


In [6]:
query = f"SELECT COUNT(*) as row_count FROM `{client.project}.{dataset_id}.sales_train_validation`"
print(client.query(query).to_dataframe())

   row_count
0      30490


In [7]:
# ---- Data Quality Checks ----

# 1. Check for nulls in key columns
null_check = f"""
SELECT
  COUNTIF(item_id IS NULL) as null_item_id,
  COUNTIF(store_id IS NULL) as null_store_id,
  COUNTIF(dept_id IS NULL) as null_dept_id
FROM `{client.project}.{dataset_id}.sales_train_validation`
"""
print(client.query(null_check).to_dataframe())

# 2. Check calendar date range and null events
calendar_check = f"""
SELECT
  MIN(date) as start_date,
  MAX(date) as end_date,
  COUNTIF(event_name_1 IS NULL) as no_event_days,
  COUNT(*) as total_days
FROM `{client.project}.{dataset_id}.calendar`
"""
print(client.query(calendar_check).to_dataframe())

# 3. Check price data for nulls/negatives
price_check = f"""
SELECT
  COUNTIF(sell_price IS NULL) as null_prices,
  COUNTIF(sell_price <= 0) as invalid_prices,
  MIN(sell_price) as min_price,
  MAX(sell_price) as max_price
FROM `{client.project}.{dataset_id}.sell_prices`
"""
print(client.query(price_check).to_dataframe())

# 4. Confirm store/category consistency
consistency_check = f"""
SELECT DISTINCT store_id, state_id
FROM `{client.project}.{dataset_id}.sales_train_validation`
ORDER BY store_id
"""
print(client.query(consistency_check).to_dataframe())

   null_item_id  null_store_id  null_dept_id
0             0              0             0
   start_date    end_date  no_event_days  total_days
0  2011-01-29  2016-06-19           1807        1969
   null_prices  invalid_prices  min_price  max_price
0            0               0       0.01     107.32
  store_id state_id
0     CA_1       CA
1     CA_2       CA
2     CA_3       CA
3     CA_4       CA
4     TX_1       TX
5     TX_2       TX
6     TX_3       TX
7     WI_1       WI
8     WI_2       WI
9     WI_3       WI
